# Orders Data Model — Interview Task

**Scenario.** Customers place orders. Each order has multiple line items (product / quantity / status). When every item is DELIVERED the order is COMPLETED. Status is tracked per line item.

This notebook runs end-to-end on **Databricks Free / Community Edition**:

1. **Bronze** — raw landing tables (SQL; all columns `STRING`)
2. **Silver** — **PySpark** + Delta `MERGE INTO`: typed rows, dedupe, FK-clean joins
3. **Gold** — **PySpark**: dims, fact, pre-aggregated marts, derived order-status view
4. **Task 2** — SQL against the monthly mart (most sold product)

**Scheduling (production shape):** two **Workflow / Lakeflow Job** tasks call `pipeline/run_silver_job.py` then `pipeline/run_gold_job.py` — see `databricks.yml`.

Keep this repo's `pipeline/` folder **next to** this notebook (Databricks Repos) so imports work.

Just hit **Run All**.


## 0. Setup — create the medallion schemas


In [ ]:
%sql
USE CATALOG dbricks_task;
CREATE SCHEMA IF NOT EXISTS orders_bronze;
CREATE SCHEMA IF NOT EXISTS orders_silver;
CREATE SCHEMA IF NOT EXISTS orders_gold;
USE orders_bronze;


## 1. Bronze — raw landing tables

All columns are `STRING`. This preserves exactly what the source sent; any typing / cleaning happens in Silver.


In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_customers (
    customer_id  STRING, first_name STRING, last_name STRING,
    email        STRING, phone      STRING, created_at STRING,
    _ingest_ts   TIMESTAMP
) USING DELTA;

CREATE OR REPLACE TABLE bronze_products (
    product_id   STRING, sku          STRING, product_name STRING,
    category     STRING, unit_price   STRING, created_at   STRING,
    _ingest_ts   TIMESTAMP
) USING DELTA;

CREATE OR REPLACE TABLE bronze_orders (
    order_id         STRING, customer_id      STRING,
    order_date       STRING, shipping_address STRING,
    _ingest_ts       TIMESTAMP
) USING DELTA;

CREATE OR REPLACE TABLE bronze_order_items (
    order_item_id     STRING, order_id   STRING, product_id STRING,
    quantity          STRING, unit_price STRING,
    line_status       STRING, status_updated_at STRING,
    _ingest_ts        TIMESTAMP
) USING DELTA;


## 2. Seed the Bronze tables

8 customers · 10 products · 25 orders · 57 line items, spanning 2026-02-05 → 2026-04-18.

"Last month" = March 2026 (when run on 2026-04-01 or later). The seed is designed so **Mechanical Keyboard** has a clear lead in March.


In [ ]:
%sql
TRUNCATE TABLE bronze_customers;
INSERT INTO bronze_customers VALUES
  ('1','Alice','Johnson','alice.j@example.com','+1-555-0101','2025-11-03 09:12:00',current_timestamp()),
  ('2','Bob','Smith','bob.smith@example.com','+1-555-0102','2025-11-14 10:44:00',current_timestamp()),
  ('3','Carol','Davis','carol.d@example.com','+1-555-0103','2025-12-02 14:03:00',current_timestamp()),
  ('4','David','Chen','david.chen@example.com','+1-555-0104','2025-12-21 16:22:00',current_timestamp()),
  ('5','Eve','Miller','eve.m@example.com','+1-555-0105','2026-01-07 08:55:00',current_timestamp()),
  ('6','Frank','Lee','frank.lee@example.com','+1-555-0106','2026-01-19 11:30:00',current_timestamp()),
  ('7','Grace','Kim','grace.kim@example.com','+1-555-0107','2026-02-02 13:17:00',current_timestamp()),
  ('8','Henry','Patel','henry.p@example.com','+1-555-0108','2026-02-15 15:48:00',current_timestamp());


In [ ]:
%sql
TRUNCATE TABLE bronze_products;
INSERT INTO bronze_products VALUES
  ('1','KB-MECH-01','Mechanical Keyboard','Electronics','129.99','2025-10-01 00:00:00',current_timestamp()),
  ('2','MS-WRL-01','Wireless Mouse','Electronics','39.99','2025-10-01 00:00:00',current_timestamp()),
  ('3','HB-USBC-01','USB-C Hub','Electronics','49.99','2025-10-01 00:00:00',current_timestamp()),
  ('4','LP-STND-01','Laptop Stand','Accessories','59.99','2025-10-01 00:00:00',current_timestamp()),
  ('5','DM-MAT-01','Desk Mat','Accessories','24.99','2025-10-01 00:00:00',current_timestamp()),
  ('6','MG-COF-01','Coffee Mug','Kitchen','14.99','2025-10-01 00:00:00',current_timestamp()),
  ('7','WB-BTL-01','Water Bottle','Kitchen','19.99','2025-10-01 00:00:00',current_timestamp()),
  ('8','NB-A5-01','Notebook A5','Stationery','9.99','2025-10-01 00:00:00',current_timestamp()),
  ('9','PN-SET-01','Pen Set','Stationery','12.99','2025-10-01 00:00:00',current_timestamp()),
  ('10','MA-DUAL-01','Monitor Arm','Accessories','89.99','2025-10-01 00:00:00',current_timestamp());


In [ ]:
%sql
TRUNCATE TABLE bronze_orders;
INSERT INTO bronze_orders VALUES
  -- February 2026
  ('1','1','2026-02-05 09:00:00','12 Oak St, Seattle WA',current_timestamp()),
  ('2','2','2026-02-10 10:30:00','44 Maple Ave, Portland OR',current_timestamp()),
  ('3','3','2026-02-15 14:15:00','7 Pine Rd, San Jose CA',current_timestamp()),
  ('4','4','2026-02-20 16:00:00','91 Elm Dr, Austin TX',current_timestamp()),
  ('5','5','2026-02-25 11:45:00','23 Cedar Ln, Denver CO',current_timestamp()),
  -- March 2026 ("last month")
  ('6','1','2026-03-02 08:20:00','12 Oak St, Seattle WA',current_timestamp()),
  ('7','2','2026-03-04 12:10:00','44 Maple Ave, Portland OR',current_timestamp()),
  ('8','3','2026-03-06 17:30:00','7 Pine Rd, San Jose CA',current_timestamp()),
  ('9','4','2026-03-08 09:45:00','91 Elm Dr, Austin TX',current_timestamp()),
  ('10','5','2026-03-10 14:00:00','23 Cedar Ln, Denver CO',current_timestamp()),
  ('11','6','2026-03-12 10:00:00','58 Birch Ct, Boston MA',current_timestamp()),
  ('12','7','2026-03-14 15:25:00','17 Spruce Way, Chicago IL',current_timestamp()),
  ('13','8','2026-03-16 11:05:00','88 Ash Blvd, Miami FL',current_timestamp()),
  ('14','1','2026-03-18 13:40:00','12 Oak St, Seattle WA',current_timestamp()),
  ('15','2','2026-03-20 09:15:00','44 Maple Ave, Portland OR',current_timestamp()),
  ('16','3','2026-03-22 16:50:00','7 Pine Rd, San Jose CA',current_timestamp()),
  ('17','4','2026-03-24 10:20:00','91 Elm Dr, Austin TX',current_timestamp()),
  ('18','5','2026-03-26 14:55:00','23 Cedar Ln, Denver CO',current_timestamp()),
  ('19','6','2026-03-28 08:30:00','58 Birch Ct, Boston MA',current_timestamp()),
  ('20','7','2026-03-30 12:00:00','17 Spruce Way, Chicago IL',current_timestamp()),
  -- April 2026
  ('21','1','2026-04-03 09:30:00','12 Oak St, Seattle WA',current_timestamp()),
  ('22','2','2026-04-08 11:20:00','44 Maple Ave, Portland OR',current_timestamp()),
  ('23','3','2026-04-12 15:45:00','7 Pine Rd, San Jose CA',current_timestamp()),
  ('24','4','2026-04-15 10:10:00','91 Elm Dr, Austin TX',current_timestamp()),
  ('25','5','2026-04-18 13:55:00','23 Cedar Ln, Denver CO',current_timestamp());


In [ ]:
%sql
TRUNCATE TABLE bronze_order_items;
INSERT INTO bronze_order_items VALUES
  ('1','1','1','1','129.99','DELIVERED','2026-02-09 15:00:00',current_timestamp()),
  ('2','1','4','1','59.99','DELIVERED','2026-02-09 15:00:00',current_timestamp()),
  ('3','1','5','2','24.99','DELIVERED','2026-02-09 15:00:00',current_timestamp()),
  ('4','2','2','2','39.99','DELIVERED','2026-02-14 12:00:00',current_timestamp()),
  ('5','2','3','1','49.99','DELIVERED','2026-02-14 12:00:00',current_timestamp()),
  ('6','3','6','4','14.99','DELIVERED','2026-02-18 17:00:00',current_timestamp()),
  ('7','3','7','2','19.99','DELIVERED','2026-02-18 17:00:00',current_timestamp()),
  ('8','3','8','3','9.99','DELIVERED','2026-02-18 17:00:00',current_timestamp()),
  ('9','4','10','1','89.99','DELIVERED','2026-02-25 10:00:00',current_timestamp()),
  ('10','4','9','2','12.99','DELIVERED','2026-02-25 10:00:00',current_timestamp()),
  ('11','5','1','1','129.99','DELIVERED','2026-03-01 14:00:00',current_timestamp()),
  ('12','5','2','1','39.99','DELIVERED','2026-03-01 14:00:00',current_timestamp()),
  -- March orders
  ('13','6','1','2','129.99','DELIVERED','2026-03-06 11:00:00',current_timestamp()),
  ('14','6','2','2','39.99','DELIVERED','2026-03-06 11:00:00',current_timestamp()),
  ('15','6','6','1','14.99','DELIVERED','2026-03-06 11:00:00',current_timestamp()),
  ('16','7','1','1','129.99','DELIVERED','2026-03-09 09:30:00',current_timestamp()),
  ('17','7','3','1','49.99','DELIVERED','2026-03-09 09:30:00',current_timestamp()),
  ('18','8','1','3','129.99','DELIVERED','2026-03-11 16:00:00',current_timestamp()),
  ('19','8','5','1','24.99','DELIVERED','2026-03-11 16:00:00',current_timestamp()),
  ('20','9','2','1','39.99','DELIVERED','2026-03-13 13:00:00',current_timestamp()),
  ('21','9','4','1','59.99','DELIVERED','2026-03-13 13:00:00',current_timestamp()),
  ('22','9','8','2','9.99','DELIVERED','2026-03-13 13:00:00',current_timestamp()),
  ('23','10','1','2','129.99','DELIVERED','2026-03-15 10:00:00',current_timestamp()),
  ('24','10','7','3','19.99','DELIVERED','2026-03-15 10:00:00',current_timestamp()),
  ('25','11','1','1','129.99','DELIVERED','2026-03-17 12:30:00',current_timestamp()),
  ('26','11','9','1','12.99','DELIVERED','2026-03-17 12:30:00',current_timestamp()),
  ('27','12','1','2','129.99','DELIVERED','2026-03-19 14:00:00',current_timestamp()),
  ('28','12','2','2','39.99','DELIVERED','2026-03-19 14:00:00',current_timestamp()),
  ('29','12','10','1','89.99','DELIVERED','2026-03-19 14:00:00',current_timestamp()),
  ('30','13','1','1','129.99','DELIVERED','2026-03-21 11:15:00',current_timestamp()),
  ('31','13','6','2','14.99','DELIVERED','2026-03-21 11:15:00',current_timestamp()),
  ('32','14','3','2','49.99','DELIVERED','2026-03-23 15:45:00',current_timestamp()),
  ('33','14','5','1','24.99','CANCELLED','2026-03-20 09:00:00',current_timestamp()),
  ('34','15','1','1','129.99','DELIVERED','2026-03-25 13:00:00',current_timestamp()),
  ('35','15','2','3','39.99','DELIVERED','2026-03-25 13:00:00',current_timestamp()),
  ('36','16','7','2','19.99','DELIVERED','2026-03-27 10:30:00',current_timestamp()),
  ('37','16','8','5','9.99','DELIVERED','2026-03-27 10:30:00',current_timestamp()),
  ('38','16','1','2','129.99','DELIVERED','2026-03-27 10:30:00',current_timestamp()),
  ('39','17','4','2','59.99','DELIVERED','2026-03-29 16:00:00',current_timestamp()),
  ('40','17','6','1','14.99','RETURNED','2026-04-02 14:00:00',current_timestamp()),
  ('41','18','1','1','129.99','DELIVERED','2026-03-31 12:00:00',current_timestamp()),
  ('42','18','2','1','39.99','DELIVERED','2026-03-31 12:00:00',current_timestamp()),
  ('43','19','9','3','12.99','DELIVERED','2026-04-02 11:00:00',current_timestamp()),
  ('44','19','1','1','129.99','SHIPPED','2026-04-01 09:00:00',current_timestamp()),
  ('45','20','10','1','89.99','DELIVERED','2026-04-04 15:30:00',current_timestamp()),
  ('46','20','2','2','39.99','DELIVERED','2026-04-04 15:30:00',current_timestamp()),
  ('47','20','3','1','49.99','DELIVERED','2026-04-04 15:30:00',current_timestamp()),
  -- April orders
  ('48','21','1','1','129.99','DELIVERED','2026-04-10 14:00:00',current_timestamp()),
  ('49','21','5','1','24.99','DELIVERED','2026-04-10 14:00:00',current_timestamp()),
  ('50','22','2','2','39.99','SHIPPED','2026-04-15 10:00:00',current_timestamp()),
  ('51','22','7','1','19.99','SHIPPED','2026-04-15 10:00:00',current_timestamp()),
  ('52','23','1','2','129.99','SHIPPED','2026-04-18 16:00:00',current_timestamp()),
  ('53','23','6','3','14.99','PENDING','2026-04-12 15:45:00',current_timestamp()),
  ('54','24','4','1','59.99','PENDING','2026-04-15 10:10:00',current_timestamp()),
  ('55','24','8','2','9.99','PENDING','2026-04-15 10:10:00',current_timestamp()),
  ('56','25','9','1','12.99','PENDING','2026-04-18 13:55:00',current_timestamp()),
  ('57','25','10','1','89.99','PENDING','2026-04-18 13:55:00',current_timestamp());


## 3. Silver — PySpark + Delta MERGE

Natural-key upserts via `MERGE INTO` (implemented in `pipeline/transforms.py`). FK integrity: orphan line items are dropped.


In [ ]:
import importlib
import os
import sys
from pathlib import Path

try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _repo_root = str(Path(_nb_path).parent)
except Exception:
    _repo_root = os.getcwd()

if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import pipeline.transforms as transforms

transforms = importlib.reload(transforms)
transforms.run_silver(spark)


## 4. Gold — PySpark (dims, fact, marts, view)

Dims + denormalized fact + daily/weekly/monthly sales aggregates + `gold_vw_order_status` (derived header status from line items).


In [ ]:
import importlib
import os
import sys
from pathlib import Path

try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _repo_root = str(Path(_nb_path).parent)
except Exception:
    _repo_root = os.getcwd()

if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import pipeline.transforms as transforms

transforms = importlib.reload(transforms)
transforms.run_gold(spark)


## 5. Task 2 — Most sold product in the last month

**Definition chosen:**
- *Sold* = line items with `line_status = 'DELIVERED'` (cancelled / pending / returned items were never truly sold).
- *Last month* = the previous **calendar** month (run on 2026-04-20 → March 2026).
- *Most sold* = highest units delivered, tie-broken by revenue, then SKU.

Expected winner on the seed data: **Mechanical Keyboard** (16 units, $2,079.84 revenue).

The primary query is a **single-row lookup against `gold_product_monthly_sales`** — no runtime aggregation. The same answer is also computed on-the-fly from the atomic fact as a sanity check. Weekly and daily variants use the other two marts.


### 5a. Primary answer — the winning product (from the monthly mart)


In [ ]:
%sql
SELECT
    product_id, sku, product_name, category,
    units_delivered   AS units_sold,
    revenue_delivered AS revenue,
    distinct_orders
FROM orders_gold.gold_product_monthly_sales
WHERE month_start = CAST(DATE_TRUNC('MONTH', ADD_MONTHS(CURRENT_DATE(), -1)) AS DATE)
ORDER BY units_delivered DESC, revenue_delivered DESC, sku
LIMIT 1;


### 5a (bis). Same answer computed on-the-fly from the atomic fact — should match


In [ ]:
%sql
WITH last_month AS (
    SELECT DATE_TRUNC('MONTH', ADD_MONTHS(CURRENT_DATE(), -1)) AS start_ts,
           DATE_TRUNC('MONTH', CURRENT_DATE())                 AS end_ts
)
SELECT f.product_id, f.sku, f.product_name, f.category,
       SUM(f.quantity)            AS units_sold,
       SUM(f.line_total)          AS revenue,
       COUNT(DISTINCT f.order_id) AS distinct_orders
FROM orders_gold.gold_fact_order_items f
CROSS JOIN last_month lm
WHERE f.order_date >= lm.start_ts
  AND f.order_date <  lm.end_ts
  AND f.line_status = 'DELIVERED'
GROUP BY f.product_id, f.sku, f.product_name, f.category
ORDER BY units_sold DESC, revenue DESC, f.sku
LIMIT 1;


### 5a (weekly). Most sold product last week (one-row lookup on the weekly mart)


In [ ]:
%sql
SELECT sku, product_name, category,
       units_delivered, revenue_delivered, distinct_orders
FROM orders_gold.gold_product_weekly_sales
WHERE week_start = CAST(DATE_TRUNC('WEEK', CURRENT_DATE() - INTERVAL 7 DAYS) AS DATE)
ORDER BY units_delivered DESC, revenue_delivered DESC
LIMIT 1;


### 5a (daily). Most sold product yesterday (one-row lookup on the daily mart)


In [ ]:
%sql
SELECT sku, product_name, category,
       units_delivered, revenue_delivered, distinct_orders
FROM orders_gold.gold_product_daily_sales
WHERE order_day = CURRENT_DATE() - INTERVAL 1 DAY
ORDER BY units_delivered DESC, revenue_delivered DESC
LIMIT 1;

-- Demo: pick a day we know has activity
SELECT sku, product_name, units_delivered, revenue_delivered
FROM orders_gold.gold_product_daily_sales
WHERE order_day = DATE '2026-03-06'
ORDER BY units_delivered DESC, revenue_delivered DESC
LIMIT 5;


### 5b. Full leaderboard (verification)


In [ ]:
%sql
WITH last_month AS (
    SELECT DATE_TRUNC('MONTH', ADD_MONTHS(CURRENT_DATE(), -1)) AS start_ts,
           DATE_TRUNC('MONTH', CURRENT_DATE())                 AS end_ts
)
SELECT f.sku, f.product_name, f.category,
       SUM(f.quantity)   AS units_sold,
       SUM(f.line_total) AS revenue
FROM orders_gold.gold_fact_order_items f
CROSS JOIN last_month lm
WHERE f.order_date >= lm.start_ts
  AND f.order_date <  lm.end_ts
  AND f.line_status = 'DELIVERED'
GROUP BY f.sku, f.product_name, f.category
ORDER BY units_sold DESC, revenue DESC;


### 5c. Alternate — trailing 30 days instead of previous calendar month


In [ ]:
%sql
SELECT f.sku, f.product_name,
       SUM(f.quantity)   AS units_sold,
       SUM(f.line_total) AS revenue
FROM orders_gold.gold_fact_order_items f
WHERE f.order_date  >= CURRENT_TIMESTAMP() - INTERVAL 30 DAYS
  AND f.line_status = 'DELIVERED'
GROUP BY f.sku, f.product_name
ORDER BY units_sold DESC, revenue DESC
LIMIT 1;


### 5d. Alternate — ranked by revenue instead of units


In [ ]:
%sql
WITH last_month AS (
    SELECT DATE_TRUNC('MONTH', ADD_MONTHS(CURRENT_DATE(), -1)) AS start_ts,
           DATE_TRUNC('MONTH', CURRENT_DATE())                 AS end_ts
)
SELECT f.sku, f.product_name,
       SUM(f.line_total) AS revenue,
       SUM(f.quantity)   AS units_sold
FROM orders_gold.gold_fact_order_items f
CROSS JOIN last_month lm
WHERE f.order_date >= lm.start_ts
  AND f.order_date <  lm.end_ts
  AND f.line_status = 'DELIVERED'
GROUP BY f.sku, f.product_name
ORDER BY revenue DESC, units_sold DESC
LIMIT 1;


## 6. Sanity check — order-level status distribution


In [ ]:
%sql
SELECT order_status, COUNT(*) AS n_orders
FROM orders_gold.gold_vw_order_status
GROUP BY order_status
ORDER BY n_orders DESC;


A peek at a few COMPLETED and PARTIALLY_SHIPPED orders:


In [ ]:
%sql
SELECT * FROM orders_gold.gold_vw_order_status
WHERE order_status IN ('COMPLETED','PARTIALLY_SHIPPED')
ORDER BY order_status, order_id
LIMIT 10;


## Done

- **Task 1** — the ERD is in `diagrams/erd.mmd`; the write-up in `data_model.md`. The model is `customer 1..N → order 1..N → order_item N..1 ← product`, with order-level status **derived** from line-item statuses (never stored, so it can never drift).
- **Task 2** — see section 5a. On this synthetic data the answer is **Mechanical Keyboard** with 16 units in March 2026.
- **PySpark** — Silver/Gold logic lives in `pipeline/transforms.py` (same semantics as `sql/03_silver_transform.sql` and `sql/04_gold_transform.sql`).
- **Jobs** — import `databricks.yml` as a bundle or create a Workflow with two **Python file** tasks pointing at `pipeline/run_silver_job.py` and `pipeline/run_gold_job.py` (Gold depends on Silver). Bronze must be loaded first.

If you want to test the derivation, toggle one line's `line_status` from DELIVERED to PENDING in `bronze_order_items`, re-run Silver → Gold, and watch the order move from COMPLETED to PARTIALLY_SHIPPED in the view.
